# Data Cleaning: SerpAPI (Google Jobs) Listings
**CSC 366: Principles of Data Science | Group 02**

This notebook cleans the Google Jobs listings collected through SerpAPI on Oct 24, 2026 (8 entry-level job titles across 8 locations, run in 4 batches because of SerpAPI's search limits). The output, `serp_clean.csv`, uses the same column names as `hn_clean.csv` so the two can be merged in the next notebook. Skills and years of experience are extracted after merging.

**Upload `raw_serpapi_batch1.json` to `raw_serpapi_batch4.json` to the Colab files panel before running.**

## Setup and loading

The searches were split into 4 batch files because of SerpAPI's rate limit,  

The first step is combining them into one table.

**AI assistance:** AI (Claude) recommended using the `*` wildcard in `glob.glob("raw_serpapi_batch*.json")`, so all 4 batch files are found and loaded automatically.

In [1]:
import pandas as pd
import numpy as np
import json
import glob

In [2]:
# Testing glob finds (should be all 4 batch files)
fileList = sorted(glob.glob("raw_serpapi_batch*.json"))
print(fileList)


['raw_serpapi_batch1.json', 'raw_serpapi_batch2.json', 'raw_serpapi_batch3.json', 'raw_serpapi_batch4.json']


**Structure of each batch file:**

```
raw_serpapi_batch1.json
 ├─ search 1  (query: "Junior Software Engineer", location: "Seattle")
 │    └─ response
 │         └─ jobs_results
 │              ├─ job 1  {title, company_name, location, extensions, ...}
 │              ├─ job 2
 │              └─ ...
 ├─ search 2
 └─ ... (48 searches)
```

Two columns are added to each job:
- `search_query`: the job title that was searched
- `search_location`: the city that was searched

These are needed later, because Google Jobs sometimes returns jobs from a different city than the one searched.

`.get()` is used instead of square brackets so that a search with no results is skipped instead of causing an error.

In [4]:
# Load the SerpAPI batch files and combine them into one DataFrame

rows = []

for path in sorted(glob.glob("raw_serpapi_batch*.json")):
    with open(path) as f:
        searches = json.load(f)              # each file is a list of 48 searches

    for search in searches:
        jobs = search.get("response", {}).get("jobs_results", [])

        for job in jobs:
            job["search_query"] = search.get("query")
            job["search_location"] = search.get("location")
            rows.append(job)

dfSerp = pd.DataFrame(rows)


dfSerp.head(3)

,title,company_name,location,via,job_highlights,share_link,thumbnail,extensions,source_link,job_title,description,apply_options,job_id,search_query,search_location,detected_extensions
0,Software Engineer (Entry Level),SynergisticIT,"New York, NY",Career.io,"[{'title': 'Qualifications', 'items': ['We're ...",https://www.google.com/search?ibp=htl;jobs&q=E...,https://serpapi.com/images/i/iVBORw0KGgoAAAANS...,"[100K a year, Full-time]",https://career.io/job/software-engineer-entry-...,Software Engineer (Entry Level),Are you passionate about coding or technology ...,"[{'title': 'Career.io', 'link': 'https://caree...",eyJqb2JfdGl0bGUiOiJTb2Z0d2FyZSBFbmdpbmVlciAoRW...,Entry Level Software Engineer,"New York, New York, United States",NaN
1,"Software Engineer I, Entry-Level Graduation Da...",DoorDash Inc,"New York, NY",Monster,"[{'title': 'Qualifications', 'items': ['A bach...",https://www.google.com/search?ibp=htl;jobs&q=E...,https://serpapi.com/images/i/iVBORw0KGgoAAAANS...,"[107K–158K a year, Full-time, Paid time off, D...",https://www.monster.com/job-openings/software-...,"Software Engineer I, Entry-Level Graduation Da...","About the Team\n\nDoorDash, Deliveroo, and Wol...","[{'title': 'Monster', 'link': 'https://www.mon...",eyJqb2JfdGl0bGUiOiJTb2Z0d2FyZSBFbmdpbmVlciBJLC...,Entry Level Software Engineer,"New York, New York, United States",NaN
2,Entry Level Software Engineer,BAE Systems,"Wayne, NJ",BAE Systems,"[{'title': 'Qualifications', 'items': ['This p...",https://www.google.com/search?ibp=htl;jobs&q=E...,NaN,"[3 days ago, Full-time, Health insurance, Dent...",https://jobs.baesystems.com/global/en/job/1313...,Entry Level Software Engineer,Job Description Our employees work on the worl...,"[{'title': 'BAE Systems', 'link': 'https://job...",eyJqb2JfdGl0bGUiOiJFbnRyeSBMZXZlbCBTb2Z0d2FyZS...,Entry Level Software Engineer,"New York, New York, United States",NaN


## First look at the data


In [6]:
print(dfSerp.shape)

dfSerp.info()

(1811, 16)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1811 entries, 0 to 1810
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   title                1811 non-null   object
 1   company_name         1811 non-null   object
 2   location             1811 non-null   object
 3   via                  1811 non-null   object
 4   job_highlights       1632 non-null   object
 5   share_link           1811 non-null   object
 6   thumbnail            1098 non-null   object
 7   extensions           1811 non-null   object
 8   source_link          1811 non-null   object
 9   job_title            1811 non-null   object
 10  description          1811 non-null   object
 11  apply_options        1811 non-null   object
 12  job_id               1811 non-null   object
 13  search_query         1811 non-null   object
 14  search_location      1811 non-null   object
 15  detected_extensions  480 non-null    object


In [7]:
# Missing value percentage check
(dfSerp.isnull().sum() / len(dfSerp) * 100).sort_values(ascending=False)

,0
detected_extensions,73.495306
thumbnail,39.370514
job_highlights,9.884042
title,0.000000
via,0.000000
location,0.000000
share_link,0.000000
company_name,0.000000
extensions,0.000000
source_link,0.000000


**Observations:**
- 1,811 job listings from 192 searches (8 job titles × 8 locations × 3 pages).
- `detected_extensions` and `thumbnail` are mostly missing, but they are not needed for the analysis.
- No salary column appears as missing, because salaries are stored inside the `extensions` column, which is explored next.

**AI assistance:** The raw SerpAPI data was shared with AI (Claude) before cleaning. Claude reviewed it and pointed out that `detected_extensions` and `thumbnail` were not needed, and that salaries were stored inside the `extensions` column rather than in a column of their own.

## Exploring the `extensions` tags

The `extensions` column holds the short tags Google attaches to each job: the salary, job type, remote status, and how long ago the job was posted, for example `["6 days ago", "105K–120K a year", "Full-time", "No degree mentioned"]`.



In [8]:
dfSerp["extensions"].head(5)

,extensions
0,"[100K a year, Full-time]"
1,"[107K–158K a year, Full-time, Paid time off, D..."
2,"[3 days ago, Full-time, Health insurance, Dent..."
3,"[Full-time, Dental insurance, Health insurance]"
4,"[6 days ago, 105K–120K a year, Full-time, No d..."


We used `.explode()` see every type of tag that exists, and to give each tag its own row. After that, `.value_counts()` was used to count each tag normally.

**AI assistance:** AI (Claude) suggested we use `.explode()` to separate the tags so they could be counted.

In [11]:
# Break each list of tags into one tag per row, then count the tags
allTags = dfSerp["extensions"].explode()

print(f"Total tags: {len(allTags)}")
allTags.value_counts().head(10)

Total tags: 5956


,count
extensions,
Full-time,1599
Health insurance,682
Dental insurance,480
No degree mentioned,480
Paid time off,468
Work from home,366
Contractor,87
3 days ago,76
4 days ago,74


**Observations:**
- The tags fall into 5 kinds: posting age (`3 days ago`), salary (`95K–154K a year`), job type (`Full-time`, `Contractor`, `Internship`, `Part-time`), remote (`Work from home`), and benefits (`Health insurance`, etc.).
- Most salary strings are unique, so salaries have to be matched by **pattern** (for example, "contains `a year`") and not by exact text.




## Job types and remote status

Use `getJobType` to identify and label job types from tags.

If no tag matches, the function returns `None`, so the value is left missing instead of being guessed.

In [12]:
# Get the job type tag (Full-time, Contractor, Internship, Part-time) from each list
def getJobType(tagList):
  jobTypes = ["Full-time", "Part-time", "Contractor", "Internship"]
  for tag in tagList:
        if tag in jobTypes:
            return tag
  return None              # no job type tag -> missing, not made up

In [13]:
dfSerp["job_type"] = dfSerp["extensions"].apply(getJobType)

dfSerp["job_type"].value_counts(dropna=False)   # dropna=False also counts missing values

,count
job_type,
Full-time,1599
Contractor,87
Internship,62
None,47
Part-time,16


In [14]:
# Remote status: True if "Work from home" is one of the tags
def isRemote(tagList):
    return "Work from home" in tagList

dfSerp["is_remote"] = dfSerp["extensions"].apply(isRemote)

dfSerp["is_remote"].value_counts()

,count
is_remote,
False,1445
True,366


## Salary Part 01

A salary tag always contains a pay unit such as `a year` or `an hour`. With strings, `in` checks whether one piece of text appears inside another (`"a year" in "105K–120K a year"` is True), so the function checks every tag against every pay unit.

In [15]:
def getSalaryText(tagList):
  payUnits = ["a year", "an hour", "a month", "a day"]
  for tag in tagList:                 # look at each tag
        for unit in payUnits:           # check it against each pay unit
            if unit in tag:             # does this tag contain this unit?
                return tag
  return None                         # no salary tag -> missing


In [16]:
# Get the salary tag (e.g. "105K–120K a year") from each list
dfSerp["salary_text"] = dfSerp["extensions"].apply(getSalaryText)

print(f"Jobs with a salary: {dfSerp['salary_text'].notna().sum()}")
dfSerp["salary_text"].head(10)

Jobs with a salary: 714


,salary_text
0,100K a year
1,107K–158K a year
2,None
3,None
4,105K–120K a year
5,None
6,None
7,95K–154K a year
8,"70,828–120,408 a year"
9,None


**Observation:** 714 of 1,811 jobs (about 39%) list a salary.



## Salary Part 02

Turning `"63.9K–97.8K a year"` into numbers

**AI assistance:** AI (Claude) was used to make sure the conversion handles every salary format found in the data (ranges, single values, "K" amounts, decimals, and commas), and to write the test lines that check each format before the code is run on the full column.


In [17]:
def toNumber(text):
    text = text.replace(",", "")

    # Check if K is in the text
    if "K" in text:
        text = text.replace("K", "")
        return float(text) * 1000

    # Otherwise, just convert to float
    return float(text)

# Test it on each format before using it on the real column
print(toNumber("63.9K"))     # should print 63900.0
print(toNumber("100K"))      # should print 100000.0
print(toNumber("70,828"))    # should print 70828.0
print(toNumber("18"))        # should print 18.0

63900.0
100000.0
70828.0
18.0


**Non-USD salaries**: Two salaries were in Canadian dollars. These rows were kept, but their salaries were left missing and labeled `non-USD`, since the analysis focuses on US salaries.

**Single values vs. ranges:** for a single value like `100K a year`, splitting on the dash gives only one piece. Using `pieces[-1]` (the last piece) for the max works for both cases: for a range it is the high end, and for a single value the min and max are the same number.

**AI assistance:** AI (Claude) generated the test cases below the `parseSalary` function, covering each salary format in the data (a range, a single value, an hourly rate, a Canadian dollar salary, and a missing salary), so the function could be checked before running it on the full column.

In [19]:
def parseSalary(salaryText):
    payUnits = ["a year", "an hour", "a month", "a day"]
    # Missing salary -> return missing for all three
    if pd.isna(salaryText):
        return None, None, None

    # Non-USD salary (e.g. "CA$70K–CA$110K a year") -> don't mix currencies
    if "CA$" in salaryText:
        return None, None, "non-USD"

    # Step 1: find which pay unit this string uses
    unit = None
    for u in payUnits:
        if u in salaryText:
            unit = u

    # Remove the unit and any extra spaces
    numberPart = salaryText.replace(unit, "").strip()

    # Step 2: split the range on the EN DASH
    pieces = numberPart.split("–")

    # Step 3: convert using the toNumber function
    salaryMin = toNumber(pieces[0])
    salaryMax = toNumber(pieces[-1])

    return salaryMin, salaryMax, unit

# Test on each format
print(parseSalary("63.9K–97.8K a year"))      # (63900.0, 97800.0, 'a year')
print(parseSalary("100K a year"))             # (100000.0, 100000.0, 'a year')
print(parseSalary("15–18 an hour"))           # (15.0, 18.0, 'an hour')
print(parseSalary("CA$70K–CA$110K a year"))   # (None, None, 'non-USD')
print(parseSalary(None))                      # (None, None, None)

(63900.0, 97800.0, 'a year')
(100000.0, 100000.0, 'a year')
(15.0, 18.0, 'an hour')
(None, None, 'non-USD')
(None, None, None)


In [20]:
# Run parseSalary on every row; each result is (min, max, unit)
salaryResults = dfSerp["salary_text"].apply(parseSalary)

# Spread the 3 values into 3 new columns
dfSerp[["salary_min", "salary_max", "salary_unit"]] = pd.DataFrame(
    salaryResults.tolist(), index=dfSerp.index
)

dfSerp["salary_unit"].value_counts(dropna=False)

,count
salary_unit,
None,1097
a year,650
an hour,53
a month,8
non-USD,2
a day,1


**Observation:** 650 yearly, 53 hourly, 8 monthly, 1 daily, 2 non-USD, and 1,097 with no salary.

## Salary Part 03

Hourly, monthly, and daily salaries were converted to yearly salaries for accurate comparison. We used a dictionary to store the conversion multipliers, and used `.map()` to apply them based on the pay unit. The converted salaries are stored in new columns, keeping the original values for reference.

In [21]:
# How much to multiply each pay unit by to get yearly pay
toYearly = {
    "a year": 1,
    "an hour": 2080,
    "a month": 12,
    "a day": 260,
}

# Look up each row's multiplier (non-USD and missing rows get NaN)
multiplier = dfSerp["salary_unit"].map(toYearly)

dfSerp["salary_min_yearly"] = dfSerp["salary_min"] * multiplier
dfSerp["salary_max_yearly"] = dfSerp["salary_max"] * multiplier

dfSerp[["salary_text", "salary_unit", "salary_min_yearly", "salary_max_yearly"]].dropna().head(15)

,salary_text,salary_unit,salary_min_yearly,salary_max_yearly
0,100K a year,a year,100000.0,100000.0
1,107K–158K a year,a year,107000.0,158000.0
4,105K–120K a year,a year,105000.0,120000.0
7,95K–154K a year,a year,95000.0,154000.0
8,"70,828–120,408 a year",a year,70828.0,120408.0
12,80K–130K a year,a year,80000.0,130000.0
13,70K a year,a year,70000.0,70000.0
15,15–18 an hour,an hour,31200.0,37440.0
16,65K–70K a year,a year,65000.0,70000.0
17,35–42 an hour,an hour,72800.0,87360.0


**Limitation:** For internships and part-time roles, the yearly column is an **annualized rate**, not actual earnings. This is why these roles will be analyzed separately from full-time roles.

**AI assistance:** AI (Claude) explained that multiplying hourly pay by 2,080 hours assumes full-time work for a whole year. Since internships usually last about 10 to 12 weeks, an intern paid \$20/hr earns about \$9,600 over a summer, not \$41,600. This is why the converted pay is treated as an annualized rate.

## Salary check

Use `.describe()` to make sure the converted salaries make sense.

In [22]:
# Summary statistics for the yearly salary columns
dfSerp[["salary_min_yearly", "salary_max_yearly"]].describe()

,salary_min_yearly,salary_max_yearly
count,712.000000,712.000000
mean,95197.965098,131151.032289
std,44616.170679,62751.183452
min,18000.000000,30000.000000
25%,70000.000000,90000.000000
50%,89500.000000,127000.000000
75%,110000.000000,154000.000000
max,600000.000000,780000.000000


**Observations:**
- The **max** (\$600,000 to \$780,000) looked like a senior position at first, but tracing the row showed it is a *Junior* Web Developer posting at TechNova Solutions listed as `50K–65K a month`.

- The **mean** (\$95,198) is higher than the **median** (\$89,500), so salaries are **right-skewed**. The median will be reported for salaries.



In [23]:
# Columns that help judge whether a salary makes sense
checkCols = ["title", "company_name", "salary_text", "job_type", "salary_min_yearly"]

# Highest 8 salaries
dfSerp.nlargest(8, "salary_min_yearly")[checkCols]

,title,company_name,salary_text,job_type,salary_min_yearly
1767,Junior Web Developer,TechNova Solutions,50K–65K a month,Full-time,600000.0
1464,Remote Software Engineer,Turing,200–300 an hour,Contractor,416000.0
1171,"Software Engineer, New Grad; Zara)",AI Trainer Jobs,130 an hour,Part-time,270400.0
460,Software Engineer - Platform / Applied AI (Ful...,JeffreyM Consulting,260K–290K a year,Full-time,260000.0
1353,Sr. to Staff Full Stack Software Engineer - Fi...,Skyrocket Ventures,250K a year,Full-time,250000.0
1360,Staff Full-Stack Engineer: Global Payments Leader,Capital One,245K–279K a year,Full-time,245000.0
884,Remote-First Junior Fullstack Engineer | API &...,"SerpApi, LLC","232,448 a year",Full-time,232448.0
1808,Remote-First Junior Fullstack Engineer | API &...,"SerpApi, LLC","232,448 a year",Full-time,232448.0


In [24]:
# Lowest 8 salaries
dfSerp.nsmallest(8, "salary_min_yearly")[checkCols]

,title,company_name,salary_text,job_type,salary_min_yearly
977,Junior Software Engineer – Optical Platform,Cisco,18K–30K a year,Full-time,18000.0
1256,Junior Software Engineer – Optical Platform,Cisco,18K–30K a year,Full-time,18000.0
515,Junior Software Developer — Paid Training & Re...,Genesis10,24K–30K a year,Full-time,24000.0
786,Junior Software Developer — Paid Training & Re...,Genesis10,24K–30K a year,Full-time,24000.0
846,Junior Software Developer — Paid Training & Re...,Genesis10,24K–30K a year,Full-time,24000.0
954,Front End Entry Level,Jewel-Osco,14.60–16.45 an hour,Part-time,30368.0
959,Front End Entry Level,Jewel-Osco,14.60–16.45 an hour,Part-time,30368.0
1015,Front End Entry Level,Jewel-Osco,14.60–16.45 an hour,Part-time,30368.0


**Types of outliers found:**

| Type | Example | Handling |
|---|---|---|
| Data error | TechNova junior role listed at `50K–65K a month` | Flag it |
| Out of scope (high) | Capital One "Staff Full-Stack Engineer" (\$245K–\$279K) | Flag with `is_senior` |
| Out of scope (low) | Jewel-Osco "Front End Entry Level," a grocery checkout job (\$14.60/hr) | Filter with `job_type` |
| Real and in scope | A new-grad role at a high-paying company | Keep it |

**Finding:** keyword searches pull in unrelated jobs at both ends of the salary range. The Staff role matched "Full Stack," and the grocery job matched "Front End" and "Entry Level," so the data has to be filtered by job title, not just by which search found it.

**AI assistance:** AI (Claude) helped look up the jobs behind the highest and lowest salaries and sort them into these outlier types.

## Flagging senior titles

This project is about entry-level jobs, so postings for senior roles need to be marked. Each job title is checked for senior-level words (senior, sr, staff, lead, principal). If one appears as a whole word, the job is flagged with `is_senior = True`. Only whole words count, so a title with "Leadership" or "Misleading" is not flagged. Senior jobs are flagged, not removed, so they can still be used for comparisons later.

**AI assistance:** AI (Claude) helped write the regex for this step and suggested which senior-level words to flag.

In [25]:
# Flag titles that signal a senior-level role (not entry-level)
seniorWords = r"\b(?:senior|sr|staff|lead|principal)\b"

dfSerp["is_senior"] = dfSerp["title"].str.contains(seniorWords, case=False, regex=True)

print(f"Senior titles flagged: {dfSerp['is_senior'].sum()}")
dfSerp[dfSerp["is_senior"]]["title"].head(20)

Senior titles flagged: 143


,title
105,QA Lead
126,"Senior Associate, Software Engineering"
131,"Senior Associate, Tax Ignition - Software Engi..."
133,Flight Simulation Software Engineer (Associate...
174,Senior Software Engineer I - Flutter
176,"Senior Software Engineer, AI (New York)"
177,"Senior Software Engineer I, Data Content"
182,Lead Software Engineer (Java/AWS) — Alternativ...
186,"Staff Software Engineer, Infrastructure Data E..."
188,"Lead Software Engineer - Python, Full Stack / AI"


**Observation:** 143 titles flagged.



## Job Post Age

Posting age tags look like `"3 days ago"`, `"1 day ago"`, or `"1 month ago"`. The first word is the number. Months are converted to days using 30 days per month (only 9 rows use months).

In [26]:
def getDaysAgo(tagList):
    for tag in tagList:
        if "ago" in tag:
            number = int(tag.split()[0])     # "3 days ago" -> 3
            if "month" in tag:
                return number * 30           # approximate months as 30 days
            return number
    return None

dfSerp["days_posted_ago"] = dfSerp["extensions"].apply(getDaysAgo)

dfSerp["days_posted_ago"].describe()

,days_posted_ago
count,968.000000
mean,11.745868
std,7.563826
min,1.000000
25%,5.000000
50%,10.000000
75%,17.000000
max,30.000000


**Observation:** 968 postings have an age. The median is 10 days and the max is 30 (`1 month ago`),  Google Jobs rarely shows postings older than about a month. The other 843 postings have no age tag and are left missing.


## Standardizing locations

The location column was messy in two ways:
- The format wasn't consistent: the same kind of information appeared as "Seattle, WA", "Massachusetts", "Anywhere", or "Hudson, NH (+2 others)", giving 305 different values.

- Second, Google Jobs treated the searched city as a suggestion, so about 30% of jobs were located somewhere other than where we searched. To fix this, each job's own location was used (not the searched city) and grouped by state.

In [28]:
# How many unique location values
print(f"Unique locations: {dfSerp['location'].nunique()}")

# Most common locations
dfSerp["location"].value_counts().head(10)

Unique locations: 305


,count
location,
Anywhere,366
"Seattle, WA",107
"New York, NY",92
"Austin, TX",84
"San Francisco, CA",74
"Chicago, IL",68
United States,46
"Boston, MA",44
"Denver, CO",43


In [29]:
# Check if job locations match the searched location
# Test Case: Boston
bostonSearch = dfSerp[dfSerp["search_location"] == "Boston, Massachusetts, United States"]
bostonSearch["location"].value_counts().head(10)

,count
location,
"Boston, MA",44
"New York, NY",23
Anywhere,17
"Middletown, RI",9
"Wakefield, MA",7
Massachusetts,6
"Quincy, MA",6
"Cambridge, MA",6
"Hartford, CT",6


**Observations:**
- There are 305 unique locations.
- The Boston search returned jobs in Massachusetts, but also in New York, Rhode Island, Connecticut, and "Anywhere" (remote).

This was confusing at first, since the searches had asked for Boston specifically. AI (Claude) explained that Google Jobs treats the search location as a suggestion, not a filter: when it runs low on nearby matches, it widens the radius. About 30% of all jobs (554) are located somewhere other than the searched city.

**Decision:** the job's own `location` is used, not `search_location`. Using the search location would count a New York job found by the Boston search as a Boston job, which would mix New York salaries into Boston's numbers.

In [30]:
# Pull the 2-letter state code from "City, ST" or "City, ST (+2 others)"
dfSerp["state"] = dfSerp["location"].str.extract(r", ([A-Z]{2})")

print(f"Still missing: {dfSerp['state'].isnull().sum()}")
dfSerp["state"].value_counts(dropna=False).head(15)

Still missing: 465


,count
state,
NaN,465
TX,199
CA,197
WA,191
CO,148
IL,135
NY,134
MA,107
NJ,69


Most locations look like `City, ST` (for example "Seattle, WA"), so the regex above pulls out the 2-letter state code.

| `location` | Becomes |
|---|---|
| `Anywhere` | Remote |
| `United States` | Unknown |
| Full state names (`Massachusetts`, `Ohio`, ...) | The 2-letter code |
| `Niskayuna` (a town in NY, no state given) | NY |
| `Indiana (+19 others)` | IN |

Endings like "(+19 others)" are removed first, so "Indiana (+19 others)" becomes "Indiana" and can be looked up. Only the rows still missing a state are filled in, so the states already found by the regex are not changed.

**Why "Unknown" instead of leaving it blank?** "Unknown" shows the location was checked and the posting just doesn't name a state. A blank could mean that, or it could mean the code missed it, and someone reading the data later couldn't tell which.

**AI assistance:** AI (Claude) helped write the regex that pulls the state code out of `City, ST` locations, and the lookup list for the locations that don't follow that format.

In [31]:
# Locations with no "City, ST" pattern -> what state they should be
locationToState = {
    "Anywhere": "Remote",
    "United States": "Unknown",
    "Niskayuna": "NY",
    "Massachusetts": "MA",
    "New Jersey": "NJ",
    "Virginia": "VA",
    "Pennsylvania": "PA",
    "Ohio": "OH",
    "New York": "NY",
    "Illinois": "IL",
    "Indiana": "IN",
    "Colorado": "CO",
    "California": "CA",
    "Washington": "WA",
    "Texas": "TX",
}

# Remove " (+2 others)" style endings so "Indiana (+19 others)" becomes "Indiana"
cleanLocation = dfSerp["location"].str.replace(r" \(\+\d+ others\)", "", regex=True)

# Look up each cleaned location in the dictionary
stateFromName = cleanLocation.map(locationToState)

# Fill ONLY the missing states with the dictionary result
dfSerp["state"] = dfSerp["state"].fillna(stateFromName)

print(f"Still missing: {dfSerp['state'].isnull().sum()}")
dfSerp["state"].value_counts().head(15)

Still missing: 0


,count
state,
Remote,366
CA,203
TX,203
WA,196
CO,151
NY,140
IL,138
MA,113
NJ,72


**Observation:** every row now has a state. Every 2-letter code captured by the regex was checked and is a real US state (no Canadian provinces slipped in).

## Duplicates

**Group Decision:** a job counts as a duplicate if it has the same **title, company, and location** as an earlier row. From a student's point of view, if a company is hiring for the same role in 4 cities, that is 4 openings. But the same remote job found by 12 different searches is still 1 opening.

`df.duplicated()` from class only catches rows where **every** column matches. That finds 0 duplicates here, because each copy of a job comes from a different search, so its `search_query` and `search_location` are different. Instead, duplicates are found using only the columns that describe the job itself.

The first copy of each job is kept as the original, and every later copy is flagged with `is_duplicate = True`.

**AI assistance:** AI (Claude) helped write the code for this step.

In [32]:
# Columns that define "the same job": same title, same company, same location
dupCols = ["title", "company_name", "location"]

# Make a copy of just these 3 columns, then lowercase each one
# so "Junior Developer" = "junior developer" (the copy keeps dfSerp unchanged)
dupCheck = dfSerp[dupCols].copy()

for col in dupCols:
    dupCheck[col] = dupCheck[col].str.lower()

# Flag every copy after the first (flag, don't delete)
dfSerp["is_duplicate"] = dupCheck.duplicated(keep="first")

print(f"Duplicates flagged: {dfSerp['is_duplicate'].sum()}")
print(f"Unique jobs: {(dfSerp['is_duplicate'] == False).sum()}")

Duplicates flagged: 579
Unique jobs: 1232


**Observation:** 579 duplicates flagged, leaving 1,232 unique jobs. About a third of the raw listings were repeats.

## Filling missing job types

47 rows had no job type tag. If the title says "intern" or "internship", those rows are labeled "Internship". Anything left is labeled "Unknown".

In [33]:
# Rows with no job type tag, and rows whose title mentions an internship
missingType = dfSerp["job_type"].isnull()
titleIsIntern = dfSerp["title"].str.contains(r"\bintern(?:ship)?\b", case=False, regex=True)

dfSerp.loc[missingType & titleIsIntern, "job_type"] = "Internship"

# Anything still missing gets labeled "Unknown"
dfSerp["job_type"] = dfSerp["job_type"].fillna("Unknown")

dfSerp["job_type"].value_counts()

,count
job_type,
Full-time,1599
Contractor,87
Internship,68
Unknown,41
Part-time,16


## Salary outlier flag



$$\text{IQR} = Q_3 - Q_1 \qquad \text{outlier if } x < Q_1 - 1.5\,\text{IQR} \ \text{ or } \ x > Q_3 + 1.5\,\text{IQR}$$

The fences are calculated only from **full-time, non-senior, non-duplicate** jobs. The flag is then applied to every row.

**AI assistance:** AI (Claude) ran the same code on the data separately and checked that the results matched the group's notebook: fences of \$2,500  and  \$182,500, with 26 salaries flagged.

In [34]:
# full-time, not senior, not a duplicate
isFullTime = dfSerp["job_type"] == "Full-time"
notSenior = dfSerp["is_senior"] == False
notDuplicate = dfSerp["is_duplicate"] == False

baseSalaries = dfSerp[isFullTime & notSenior & notDuplicate]["salary_min_yearly"]

Q1 = baseSalaries.quantile(0.25)
Q3 = baseSalaries.quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print(f"Q1 = {Q1}, Q3 = {Q3}, IQR = {IQR}")
print(f"fences: {lower} and {upper}")

# Flag (not delete) every row outside the fences
tooLow = dfSerp["salary_min_yearly"] < lower
tooHigh = dfSerp["salary_min_yearly"] > upper
dfSerp["is_salary_outlier"] = tooLow | tooHigh

print(f"Salary outliers flagged: {dfSerp['is_salary_outlier'].sum()}")
dfSerp[dfSerp["is_salary_outlier"]][["title", "company_name", "salary_text", "job_type", "is_senior"]]

Q1 = 70000.0, Q3 = 115000.0, IQR = 45000.0
fences: 2500.0 and 182500.0
Salary outliers flagged: 26


,title,company_name,salary_text,job_type,is_senior
78,"Software Engineer, ML Infrastructure",Rebar,200K–260K a year,Full-time,False
454,Product (Full Stack) Engineer | AI & Creative ...,Big Wave Digital,225K–275K a year,Full-time,False
455,All Levels: Fullstack Software Engineer,Hadrian Automation,220K–320K a year,Full-time,False
460,Software Engineer - Platform / Applied AI (Ful...,JeffreyM Consulting,260K–290K a year,Full-time,False
665,Staff Software Engineer - AI Platform,General Motors,189K–333K a year,Full-time,True
874,Staff Full-Stack Engineer: Drive New Consumer ...,Stripe,224K–336K a year,Full-time,True
884,Remote-First Junior Fullstack Engineer | API &...,"SerpApi, LLC","232,448 a year",Full-time,False
897,Principal Full-Stack Engineer: Secure Platform...,Dimensional Fund Advisors,210K–260K a year,Full-time,True
1089,"Software Engineer, Product",Meta,"183,997–257,000 a year",Full-time,False
1171,"Software Engineer, New Grad; Zara)",AI Trainer Jobs,130 an hour,Part-time,False


**Observation:** the fences are \$2,500 and \$182,500, and 26 rows are flagged.

The lower fence is below every salary in the data, so low-paying out-of-scope jobs (like the Jewel-Osco cashier role) are not caught by this flag. Those are handled by filtering on `job_type` instead.

## Save

Only the columns needed for analysis are kept, in a readable order. A `source` column is added so rows can still be told apart after merging with the Hacker News data.

**AI assistance:** AI (Claude) helped choose which columns to keep and how to name them, so that `serp_clean.csv` and `hn_clean.csv` would use the same column names, and checked both files before merging.

In [ ]:
# Keep only the columns needed for analysis, in a readable order
keepCols = ["job_id", "title", "company_name", "location", "state", "is_remote",
            "job_type", "is_senior", "salary_text", "salary_unit",
            "salary_min", "salary_max", "salary_min_yearly", "salary_max_yearly",
            "is_salary_outlier", "days_posted_ago", "is_duplicate",
            "search_query", "search_location", "via", "description"]

dfSerpClean = dfSerp[keepCols].copy()
dfSerpClean.insert(0, "source", "serpapi")     # so rows can be told apart after merging with HN

# Save to a NEW file (never overwrite raw data)
dfSerpClean.to_csv("serp_clean.csv", index=False)

print(dfSerpClean.shape)
dfSerpClean.isnull().sum()

(1811, 22)


,0
source,0
job_id,0
title,0
company_name,0
location,0
state,0
is_remote,0
job_type,0
is_senior,0
salary_text,1097
